# 07 Anomaly Forecast

Flood-It! retention and churn project. Run cells top to bottom.

### Imports and settings

In [ ]:
import pandas as pd                                                    # tables
import numpy as np                                                     # maths
import matplotlib.pyplot as plt                                        # charts
from statsmodels.tsa.seasonal import STL                               # splits a series into trend + weekly pattern + leftover
from statsmodels.tsa.exponential_smoothing.ets import ETSModel         # exponential smoothing forecast with intervals
from pathlib import Path                                               # file paths

pd.set_option("display.max_columns", None)                             # never hide columns
pd.set_option("display.width", 200)                                    # wide printing

### Paths and a clean daily series

In [ ]:
ROOT = Path.cwd()                                                      # folder the notebook runs in
if ROOT.name == "notebooks":                                           # if inside notebooks/...
    ROOT = ROOT.parent                                                 # ...go up to the project root
DATA = ROOT / "data" / "processed"                                     # data folder
FIG = ROOT / "reports" / "figures"                                     # charts folder
kpi = pd.read_csv(DATA / "kpi_daily.csv", parse_dates=["day"]).sort_values("day").set_index("day")   # one row per day
for col in kpi.columns:                                                # one loop, one column per iteration
    if col != "has_full_28d_window":                                   # that column is a True/False flag
        kpi[col] = pd.to_numeric(kpi[col], errors="coerce")            # make sure every metric is a number, not text
RAMP_UP_DAYS = 14                                                       # the first days of any export are a ramp-up, not normal behaviour
dau = kpi["dau"].iloc[RAMP_UP_DAYS:].asfreq("D")                        # drop the ramp-up, then make sure every calendar day exists
print("Missing days filled by interpolation:", int(dau.isna().sum()))  # should usually be 0
dau = dau.interpolate()                                                # fill any gap with a straight line
print(dau.describe().round(1))                                         # quick summary

### STL decomposition with a 7-day season

In [ ]:
stl = STL(dau, period=7, robust=True).fit()                            # robust = unusual days do not distort the trend
fig = stl.plot()                                                       # four panels: observed, trend, season, residual
fig.set_size_inches(12, 8)                                             # bigger chart
fig.tight_layout()                                                     # tidy
fig.savefig(FIG / "19_stl_decomposition.png", dpi=150)                 # save
plt.show()                                                             # display

### Flag anomalies with a robust z-score on the residual

In [ ]:
resid = stl.resid                                                      # what the trend and weekly pattern cannot explain
mad = np.median(np.abs(resid - np.median(resid)))                      # median absolute deviation (robust spread)
robust_z = (resid - np.median(resid)) / (1.4826 * mad)                 # 1.4826 makes MAD comparable to a standard deviation
anomalies = pd.DataFrame({"dau": dau, "expected": stl.trend + stl.seasonal, "robust_z": robust_z})   # observed vs expected
anomalies = anomalies[anomalies["robust_z"].abs() > 3.5]               # common cut-off for strong outliers
print(f"Analysis window: {dau.index.min().date()} to {dau.index.max().date()} ({len(dau)} days, first {RAMP_UP_DAYS} days excluded as ramp-up)")
print(f"Flagged {len(anomalies)} of {len(dau)} days")                   # how many days were unusual
print(anomalies.round(2))                                              # the flagged days
fig, ax = plt.subplots(figsize=(12, 4))                                # empty chart
ax.plot(dau.index, dau.values, label="DAU")                            # series
ax.plot(dau.index, stl.trend + stl.seasonal, linestyle="--", label="Expected (trend + weekly pattern)")   # expected level
ax.scatter(anomalies.index, anomalies["dau"], color="red", zorder=3, s=60, label="Anomaly (|robust z| > 3.5)")   # red dots
ax.legend()                                                            # legend
ax.set_title("DAU anomalies")                                          # title
fig.tight_layout()                                                     # tidy
fig.savefig(FIG / "20_dau_anomalies.png", dpi=150)                     # save
plt.show()                                                             # display

### Diagnose each anomaly: tracking problem or real behaviour change?

In [ ]:
ratio_cols = ["sessions_per_dau", "minutes_per_dau", "levels_per_dau"]  # per-player behaviour
typical = kpi[ratio_cols].median()                                     # normal level of each ratio
for day in anomalies.index:                                            # one loop, one anomaly per iteration
    ratios = (kpi.loc[day, ratio_cols].astype(float) / typical).round(2)   # this day relative to normal (1.0 = normal); astype keeps it numeric
    print(day.date(), "| DAU vs expected:", round(anomalies.loc[day, "dau"] / anomalies.loc[day, "expected"], 2), "| per-player ratios vs normal:", ratios.to_dict())
print("Reading: DAU far below expected while per-player ratios stay near 1.0 usually means lost data (tracking), not players behaving differently.")

### Backtest: forecast the last 14 days and compare with a seasonal naive baseline

In [ ]:
clean = dau.copy()                                                     # copy of the series
clean.loc[anomalies.index] = (stl.trend + stl.seasonal).loc[anomalies.index]   # replace anomalies with expected values before modeling
horizon = 14                                                           # days to forecast
train, test = clean.iloc[:-horizon], clean.iloc[-horizon:]             # hold out the final two weeks
naive = clean.shift(7).iloc[-horizon:]                                 # baseline: same weekday last week
ets = ETSModel(train, error="add", trend="add", damped_trend=True, seasonal="add", seasonal_periods=7).fit(disp=False)   # fit on train
ets_fc = ets.forecast(horizon)                                         # predict the held-out days
mape_naive = np.mean(np.abs((test - naive) / test)) * 100              # average % error of the baseline
mape_ets = np.mean(np.abs((test - ets_fc.values) / test)) * 100        # average % error of ETS
mae_naive = np.mean(np.abs(test - naive))                              # average absolute error of the baseline
mae_ets = np.mean(np.abs(test - ets_fc.values))                        # average absolute error of ETS
print(f"Seasonal naive: MAPE {mape_naive:.2f}%, MAE {mae_naive:.1f} | ETS: MAPE {mape_ets:.2f}%, MAE {mae_ets:.1f}")
fig, ax = plt.subplots(figsize=(12, 4))                                # empty chart
ax.plot(train.index[-42:], train.iloc[-42:], label="Train (last 6 weeks)")   # recent history
ax.plot(test.index, test, label="Actual", color="black")               # truth
ax.plot(test.index, naive, label="Seasonal naive", linestyle=":")      # baseline
ax.plot(test.index, ets_fc.values, label="ETS forecast", linestyle="--")   # model
ax.legend()                                                            # legend
ax.set_title("Backtest on the final 14 days")                          # title
fig.tight_layout()                                                     # tidy
fig.savefig(FIG / "21_forecast_backtest.png", dpi=150)                 # save
plt.show()                                                             # display

### Final 14-day forecast with 95% prediction intervals

In [ ]:
final_model = ETSModel(clean, error="add", trend="add", damped_trend=True, seasonal="add", seasonal_periods=7).fit(disp=False)   # refit on all data
start = clean.index[-1] + pd.Timedelta(days=1)                         # first future day
end = clean.index[-1] + pd.Timedelta(days=horizon)                     # last future day
future = final_model.get_prediction(start=start, end=end).summary_frame(alpha=0.05)   # mean and 95% interval
print(future.round(1))                                                 # table
fig, ax = plt.subplots(figsize=(12, 4))                                # empty chart
ax.plot(clean.index[-56:], clean.iloc[-56:], label="History (last 8 weeks)")   # history
ax.plot(future.index, future["mean"], label="Forecast", color="tab:red")        # forecast line
ax.fill_between(future.index, future["pi_lower"], future["pi_upper"], color="tab:red", alpha=0.2, label="95% prediction interval")   # uncertainty band
ax.legend()                                                            # legend
ax.set_title("DAU forecast for the next 14 days")                      # title
fig.tight_layout()                                                     # tidy
fig.savefig(FIG / "22_dau_forecast.png", dpi=150)                      # save
plt.show()                                                             # display
future.to_csv(DATA / "dau_forecast.csv")                               # save forecast
anomalies.to_csv(DATA / "dau_anomalies.csv")                           # save anomalies